In [1]:
import os
import itertools
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
# Residual Block Models defination generator and discriminator
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.InstanceNorm2d(channels),
            nn.ReLU(True),
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)

# Generator
class Generator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, n_residuals=6):
        super().__init__()
        model = [
            nn.Conv2d(in_channels, 64, 7, 1, 3),
            nn.InstanceNorm2d(64),
            nn.ReLU(True),
        ]
        # Downsampling
        in_features = 64
        for _ in range(2):
            model += [
                nn.Conv2d(in_features, in_features * 2, 3, 2, 1),
                nn.InstanceNorm2d(in_features * 2),
                nn.ReLU(True)
            ]
            in_features *= 2
        # Residuals
        for _ in range(n_residuals):
            model += [ResidualBlock(in_features)]
        # Upsampling
        for _ in range(2):
            model += [
                nn.ConvTranspose2d(in_features, in_features // 2, 3, 2, 1, output_padding=1),
                nn.InstanceNorm2d(in_features // 2),
                nn.ReLU(True)
            ]
            in_features = in_features // 2
        # Output
        model += [nn.Conv2d(64, out_channels, 7, 1, 3), nn.Tanh()]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)

# Discriminator
class Discriminator(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        def block(in_feat, out_feat, norm=True):
            layers = [nn.Conv2d(in_feat, out_feat, 4, 2, 1)]
            if norm:
                layers.append(nn.InstanceNorm2d(out_feat))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(in_channels, 64, norm=False),
            *block(64, 128),
            *block(128, 256),
            *block(256, 512),
            nn.Conv2d(512, 1, 4, padding=1)
        )

    def forward(self, x):
        return self.model(x)


In [3]:
class ImageDataset(Dataset):
    def __init__(self, rootA, rootB, transform):
        self.filesA = sorted(os.listdir(rootA))
        self.filesB = sorted(os.listdir(rootB))
        self.rootA = rootA
        self.rootB = rootB
        self.transform = transform

    def __len__(self):
        return min(len(self.filesA), len(self.filesB))

    def __getitem__(self, idx):
        imgA = Image.open(os.path.join(self.rootA, self.filesA[idx % len(self.filesA)])).convert("RGB")
        imgB = Image.open(os.path.join(self.rootB, self.filesB[idx % len(self.filesB)])).convert("RGB")
        return self.transform(imgA), self.transform(imgB)


In [4]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = ImageDataset("Augmented_Average_Bedroom", "Augmented_Modern_Bedroom", transform)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)


In [5]:
G_AB = Generator().to(device)
G_BA = Generator().to(device)
D_A = Discriminator().to(device)
D_B = Discriminator().to(device)

criterion_GAN = nn.MSELoss()
criterion_cycle = nn.L1Loss()

optimizer_G = optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_A = optim.Adam(D_A.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_B = optim.Adam(D_B.parameters(), lr=0.0002, betas=(0.5, 0.999))


In [6]:
import os
import time
import torch
import pandas as pd
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

# Setup for saving
save_path = "saved_models_cyclegans"
os.makedirs(save_path, exist_ok=True)

# FID & IS metrics setup
fid_metric = FrechetInceptionDistance(feature=64).to(device)
inception_metric = InceptionScore().to(device)

csv_path = "training_metrics.csv"
if not os.path.exists(csv_path):
    pd.DataFrame(columns=["epoch", "fid", "inception", "eta_secs"]).to_csv(csv_path, index=False)

# 🔧 Fix: convert float32 [-1,1] → uint8 [0,255]
def denormalize_to_uint8(tensor):
    tensor = (tensor * 0.5 + 0.5) * 255.0
    return tensor.clamp(0, 255).to(torch.uint8)


# Training loop
for epoch in range(1, 1000):  # recommended: 200 or 500

    start_time = time.time()

    for i, (real_A, real_B) in enumerate(dataloader):
        real_A = real_A.to(device)
        real_B = real_B.to(device)

        # ---- Train Generators ----
        optimizer_G.zero_grad()

        fake_B = G_AB(real_A)
        pred_fake_B = D_B(fake_B)
        loss_GAN_AB = criterion_GAN(pred_fake_B, torch.ones_like(pred_fake_B))

        fake_A = G_BA(real_B)
        pred_fake_A = D_A(fake_A)
        loss_GAN_BA = criterion_GAN(pred_fake_A, torch.ones_like(pred_fake_A))

        recovered_A = G_BA(fake_B)
        recovered_B = G_AB(fake_A)

        loss_cycle_A = criterion_cycle(recovered_A, real_A)
        loss_cycle_B = criterion_cycle(recovered_B, real_B)

        loss_G = loss_GAN_AB + loss_GAN_BA + 10 * (loss_cycle_A + loss_cycle_B)
        loss_G.backward()
        optimizer_G.step()

        # ---- Train Discriminator A ----
        optimizer_D_A.zero_grad()
        loss_real_A = criterion_GAN(D_A(real_A), torch.ones_like(D_A(real_A)))
        loss_fake_A = criterion_GAN(D_A(fake_A.detach()), torch.zeros_like(D_A(fake_A)))
        loss_D_A = (loss_real_A + loss_fake_A) * 0.5
        loss_D_A.backward()
        optimizer_D_A.step()

        # ---- Train Discriminator B ----
        optimizer_D_B.zero_grad()
        loss_real_B = criterion_GAN(D_B(real_B), torch.ones_like(D_B(real_B)))
        loss_fake_B = criterion_GAN(D_B(fake_B.detach()), torch.zeros_like(D_B(fake_B)))
        loss_D_B = (loss_real_B + loss_fake_B) * 0.5
        loss_D_B.backward()
        optimizer_D_B.step()

        torch.cuda.empty_cache()  # Free VRAM

        if i % 50 == 0:
            print(f"[Epoch {epoch}] [Batch {i}] "
                  f"D_A: {loss_D_A.item():.4f} | D_B: {loss_D_B.item():.4f} | G: {loss_G.item():.4f}")

    # ---- Metrics ----
    fid_metric.reset()
    inception_metric.reset()
    for real_A, real_B in dataloader:
        real_A = real_A.to(device)
        real_B = real_B.to(device)
        with torch.no_grad():
            fake_B = G_AB(real_A)

        fake_B_uint8 = denormalize_to_uint8(fake_B)
        real_B_uint8 = denormalize_to_uint8(real_B)

        fid_metric.update(fake_B_uint8, real=False)
        fid_metric.update(real_B_uint8, real=True)
        inception_metric.update(fake_B_uint8)

    fid_score = fid_metric.compute().item()
    inception_score = inception_metric.compute()[0].item()
    epoch_time = int(time.time() - start_time)

    # ---- Save metrics ----
    df = pd.read_csv(csv_path)
    df.loc[len(df)] = [epoch, fid_score, inception_score, epoch_time]
    df.to_csv(csv_path, index=False)

    print(f"📊 Epoch {epoch} | FID: {fid_score:.2f} | IS: {inception_score:.2f} | ETA: {epoch_time}s")

    # ---- Save models ----
    if epoch % 5 == 0:
        torch.save(G_AB.state_dict(), f"{save_path}/G_AB_epoch{epoch}.pth")
        torch.save(G_BA.state_dict(), f"{save_path}/G_BA_epoch{epoch}.pth")
        torch.save(D_A.state_dict(), f"{save_path}/D_A_epoch{epoch}.pth")
        torch.save(D_B.state_dict(), f"{save_path}/D_B_epoch{epoch}.pth")
        print(f"✅ Models saved at epoch {epoch}")

print("🚀 Training completed!")


/home/student/anaconda3/envs/pytorch/lib/python3.11/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Metric `InceptionScore` will save all extracted features in buffer. For large datasets this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)


[Epoch 1] [Batch 0] D_A: 0.8836 | D_B: 0.6619 | G: 13.6677
[Epoch 1] [Batch 50] D_A: 0.2417 | D_B: 0.2162 | G: 7.8798


KeyboardInterrupt: 

In [ ]:
#started training from epoch 65




import os
import time
import torch
import pandas as pd
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore

# Assumed: These are already defined somewhere above this script
# device, dataloader, G_AB, G_BA, D_A, D_B, 
# criterion_GAN, criterion_cycle, 
# optimizer_G, optimizer_D_A, optimizer_D_B

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "saved_models_cyclegans"
os.makedirs(save_path, exist_ok=True)

# FID & IS setup
fid_metric = FrechetInceptionDistance(feature=64).to(device)
inception_metric = InceptionScore().to(device)

csv_path = "training_metrics.csv"
if not os.path.exists(csv_path):
    pd.DataFrame(columns=["epoch", "fid", "inception", "eta_secs"]).to_csv(csv_path, index=False)

# Denormalization helper
def denormalize_to_uint8(tensor):
    tensor = (tensor * 0.5 + 0.5) * 255.0
    return tensor.clamp(0, 255).to(torch.uint8)

# Resume from epoch 66
start_epoch = 60

# Load checkpoints
G_AB.load_state_dict(torch.load(f"{save_path}/G_AB_epoch60.pth"))
G_BA.load_state_dict(torch.load(f"{save_path}/G_BA_epoch60.pth"))
D_A.load_state_dict(torch.load(f"{save_path}/D_A_epoch60.pth"))
D_B.load_state_dict(torch.load(f"{save_path}/D_B_epoch60.pth"))

G_AB.to(device)
G_BA.to(device)
D_A.to(device)
D_B.to(device)

# Training loop
for epoch in range(start_epoch, 1000):
    start_time = time.time()

    for i, (real_A, real_B) in enumerate(dataloader):
        real_A = real_A.to(device)
        real_B = real_B.to(device)

        # Train Generators
        optimizer_G.zero_grad()

        fake_B = G_AB(real_A)
        pred_fake_B = D_B(fake_B)
        loss_GAN_AB = criterion_GAN(pred_fake_B, torch.ones_like(pred_fake_B))

        fake_A = G_BA(real_B)
        pred_fake_A = D_A(fake_A)
        loss_GAN_BA = criterion_GAN(pred_fake_A, torch.ones_like(pred_fake_A))

        recovered_A = G_BA(fake_B)
        recovered_B = G_AB(fake_A)

        loss_cycle_A = criterion_cycle(recovered_A, real_A)
        loss_cycle_B = criterion_cycle(recovered_B, real_B)

        loss_G = loss_GAN_AB + loss_GAN_BA + 10 * (loss_cycle_A + loss_cycle_B)
        loss_G.backward()
        optimizer_G.step()

        # Train Discriminator A
        optimizer_D_A.zero_grad()
        loss_real_A = criterion_GAN(D_A(real_A), torch.ones_like(D_A(real_A)))
        loss_fake_A = criterion_GAN(D_A(fake_A.detach()), torch.zeros_like(D_A(fake_A)))
        loss_D_A = (loss_real_A + loss_fake_A) * 0.5
        loss_D_A.backward()
        optimizer_D_A.step()

        # Train Discriminator B
        optimizer_D_B.zero_grad()
        loss_real_B = criterion_GAN(D_B(real_B), torch.ones_like(D_B(real_B)))
        loss_fake_B = criterion_GAN(D_B(fake_B.detach()), torch.zeros_like(D_B(fake_B)))
        loss_D_B = (loss_real_B + loss_fake_B) * 0.5
        loss_D_B.backward()
        optimizer_D_B.step()

        torch.cuda.empty_cache()

        if i % 50 == 0:
            print(f"[Epoch {epoch}] [Batch {i}] "
                  f"D_A: {loss_D_A.item():.4f} | D_B: {loss_D_B.item():.4f} | G: {loss_G.item():.4f}")

    # ---- Metrics ----
    fid_metric.reset()
    inception_metric.reset()

    for real_A, real_B in dataloader:
        real_A = real_A.to(device)
        real_B = real_B.to(device)
        with torch.no_grad():
            fake_B = G_AB(real_A)

        fake_B_uint8 = denormalize_to_uint8(fake_B)
        real_B_uint8 = denormalize_to_uint8(real_B)

        fid_metric.update(fake_B_uint8, real=False)
        fid_metric.update(real_B_uint8, real=True)
        inception_metric.update(fake_B_uint8)

    fid_score = fid_metric.compute().item()
    inception_score = inception_metric.compute()[0].item()
    epoch_time = int(time.time() - start_time)

    # Save metrics
    df = pd.read_csv(csv_path)
    df.loc[len(df)] = [epoch, fid_score, inception_score, epoch_time]
    df.to_csv(csv_path, index=False)

    print(f"📊 Epoch {epoch} | FID: {fid_score:.2f} | IS: {inception_score:.2f} | ETA: {epoch_time}s")

    # Save models
    if epoch % 5 == 0:
        torch.save(G_AB.state_dict(), f"{save_path}/G_AB_epoch{epoch}.pth")
        torch.save(G_BA.state_dict(), f"{save_path}/G_BA_epoch{epoch}.pth")
        torch.save(D_A.state_dict(), f"{save_path}/D_A_epoch{epoch}.pth")
        torch.save(D_B.state_dict(), f"{save_path}/D_B_epoch{epoch}.pth")
        print(f"✅ Models saved at epoch {epoch}")

print("🚀 Training completed!")


[Epoch 60] [Batch 0] D_A: 0.1327 | D_B: 0.1364 | G: 2.1509
[Epoch 60] [Batch 50] D_A: 0.0404 | D_B: 0.1158 | G: 2.4947
[Epoch 60] [Batch 100] D_A: 0.1035 | D_B: 0.1531 | G: 2.2336
[Epoch 60] [Batch 150] D_A: 0.0843 | D_B: 0.0986 | G: 2.1927
[Epoch 60] [Batch 200] D_A: 0.1093 | D_B: 0.0871 | G: 2.5529


In [ ]:
def show_tensor_image(tensor):
    image = tensor.detach().cpu().squeeze(0)
    image = (image * 0.5) + 0.5  # unnormalize
    plt.imshow(image.permute(1, 2, 0))
    plt.axis('off')
    plt.show()

real_A, real_B = next(iter(dataloader))
real_A = real_A.to(device)
fake_B = G_AB(real_A)
show_tensor_image(fake_B)
